In [ ]:
import pandas as pd
import kagglehub

# 1. Descargar el dataset de Olist
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

# 2. Cargar las tablas principales que utilizaremos
orders = pd.read_csv(f"{path}/olist_orders_dataset.csv")
customers = pd.read_csv(f"{path}/olist_customers_dataset.csv")
order_items = pd.read_csv(f"{path}/olist_order_items_dataset.csv")
order_reviews = pd.read_csv(f"{path}/olist_order_reviews_dataset.csv")

In [3]:
# Convertimos a datetime
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

# Filtramos solo compras efectivas
orders_valid = orders[orders['order_status'] == 'delivered'].copy()

In [14]:
# 1. Unimos órdenes con clientes para llegar al id único
df = orders_valid.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id', how='inner')

# 2. Unimos con los ítems de las órdenes
df = df.merge(order_items[['order_id', 'price', 'freight_value']], on='order_id', how='left')

# 3. Unimos con las reseñas
df = df.merge(order_reviews[['order_id', 'review_score']], on='order_id', how='left')

In [15]:
# Definimos la fecha de referencia del dataset (la fecha maxima en Olist)
fecha_referencia = df['order_purchase_timestamp'].max()

# Agrupamos por cliente único, calculando por columna según lo que necesitamos.
df_churn = df.groupby('customer_unique_id').agg(
    ultima_compra=('order_purchase_timestamp', 'max'),
    total_compras=('order_id', 'nunique'),
    monto_total_gastado=('price', 'sum'),
    promedio_flete_pagado=('freight_value', 'mean'),
    promedio_review_score=('review_score', 'mean')
).reset_index()

# Calculamos los días sin comprar contra el presente
dias_inactivo = (fecha_referencia - df_churn['ultima_compra']).dt.days

# Definimos churn según nuestra logica de negocio si paso 90 días de compra es churn
df_churn['churn'] = (dias_inactivo > 90).astype(int)

In [19]:
# 1. Calculamos la nota promedio global de todas las reseñas válidas en la plataforma
media_review_global = df['review_score'].mean()

# 2. Construimos la matriz de características X imputando con la media global en las reseñas
X = df_churn[['total_compras', 'monto_total_gastado', 'promedio_flete_pagado', 'promedio_review_score']].copy()

# Rellenamos nulos de review con la media global
X['promedio_review_score'] = X['promedio_review_score'].fillna(media_review_global)

# La variable objetivo y
y = df_churn['churn']

print("Dimensiones de X:", X.shape)
print("\nDistribución de la variable objetivo (y):")
print(y.value_counts(normalize=True))

Dimensiones de X: (93358, 4)

Distribución de la variable objetivo (y):
churn
1    0.800874
0    0.199126
Name: proportion, dtype: float64


In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f"Filas en Train: {len(X_train)} | Filas en Test: {len(X_test)}")

Filas en Train: 74686 | Filas en Test: 18672
